# QQKRLS Complete Tutorial — From Theory to Publication

## Quantile-on-Quantile Kernel-Based Regularized Least Squares

**Author**: Dr. Merwan Roudane  
**Email**: merwanroudane920@gmail.com  
**GitHub**: [github.com/merwanroudane/qqkrls](https://github.com/merwanroudane/qqkrls)  
**PyPI**: [pypi.org/project/qqkrls](https://pypi.org/project/qqkrls/)  

---

This notebook provides a **comprehensive, step-by-step guide** to the `qqkrls` Python library,
covering all four core methodologies with real-world economic data:

| # | Method | Description |
|---|--------|-------------|
| 1 | **KRLS** | Kernel Regularized Least Squares — nonparametric regression |
| 2 | **QQ Regression** | Quantile-on-Quantile analysis of distributional effects |
| 3 | **WQR** | Weighted Quantile Regression (via the `wqr` legacy) |
| 4 | **QQKRLS** | Combined estimator — nonparametric marginal effects across all quantile pairs |

### Table of Contents

1. [Installation & Setup](#1-installation--setup)
2. [Theoretical Background](#2-theoretical-background)
3. [Data: Real Economic Series from FRED](#3-data-real-economic-series-from-fred)
4. [Descriptive Statistics](#4-descriptive-statistics)
5. [Pre-Estimation Diagnostics](#5-pre-estimation-diagnostics)
6. [KRLS Estimation](#6-krls-estimation)
7. [KRLS Marginal Effects & Visualization](#7-krls-marginal-effects--visualization)
8. [KRLS Post-Estimation Diagnostics](#8-krls-post-estimation-diagnostics)
9. [QQKRLS Estimation (Single Variable)](#9-qqkrls-estimation-single-variable)
10. [QQKRLS Heatmap & 3D Surface](#10-qqkrls-heatmap--3d-surface)
11. [QQKRLS Contour & P-Value Maps](#11-qqkrls-contour--p-value-maps)
12. [Multi-Variable QQKRLS Panel](#12-multi-variable-qqkrls-panel)
13. [LaTeX Tables & CSV Export](#13-latex-tables--csv-export)
14. [Interpretation Guide](#14-interpretation-guide)
15. [References](#15-references)

---
## 1. Installation & Setup

Install the `qqkrls` package from PyPI and import all required modules.

In [ ]:
# Install the package (uncomment if not already installed)
# !pip install qqkrls pandas-datareader

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── Import ALL qqkrls components ──
from qqkrls import (
    # Core estimation
    krls, KRLSResult, predict_krls, gaussian_kernel,
    qqkrls, QQKRLSResult,
    # Visualization
    plot_qqkrls_heatmap, plot_qqkrls_3d, plot_qqkrls_contour,
    plot_qqkrls_pvalue, plot_qqkrls_panel,
    plot_krls_derivatives, plot_krls_fit, plot_krls_panel,
    # Tables
    qqkrls_coefficient_table, krls_summary_table,
    descriptive_statistics, export_results_csv,
    # Diagnostics
    bds_test, parameter_stability_test, jarque_bera,
    linearity_test_battery, print_diagnostics,
    krls_residual_diagnostics, print_krls_diagnostics,
    multi_qqkrls,
)

print('qqkrls imported successfully!')
from qqkrls._version import __version__
print(f'Version: {__version__}')

---
## 2. Theoretical Background

### 2.1 Kernel Regularized Least Squares (KRLS)

KRLS (Hainmueller & Hazlett, 2014) is a machine-learning regression method that fits a flexible,
nonparametric function $f$ in a reproducing kernel Hilbert space (RKHS).

**Objective function:**

$$\hat{f} = \arg\min_{f \in \mathcal{H}_K} \sum_{i=1}^{N} (y_i - f(x_i))^2 + \lambda \|f\|_{\mathcal{H}_K}^2$$

By the representer theorem, the solution has the form:

$$\hat{f}(x) = \sum_{i=1}^{N} c_i \cdot K(x, x_i)$$

where $K$ is a **Gaussian kernel**:

$$K(x_i, x_j) = \exp\!\left(-\frac{\|x_i - x_j\|^2}{\sigma}\right)$$

The coefficient vector is obtained via eigendecomposition:

$$c^* = V (\Lambda + \lambda I)^{-1} V^\top y$$

where $K = V \Lambda V^\top$.

**Key features:**
- No linearity or additivity assumptions
- Automatic bandwidth ($\sigma = D$, number of predictors)
- LOO cross-validation for regularization parameter $\lambda$
- Closed-form pointwise marginal effects

**Pointwise marginal effects:**

$$\frac{\partial \hat{f}(x_j)}{\partial x_j^{(d)}} = \frac{-2}{\sigma} \sum_{i=1}^{N} c_i \cdot K(x_j, x_i) \cdot (x_i^{(d)} - x_j^{(d)})$$

### 2.2 Quantile-on-Quantile (QQ) Regression

QQ regression (Sim & Zhou, 2015) examines how the $\theta$-th quantile of $X$ affects
the $\tau$-th conditional quantile of $Y$. This creates a **two-dimensional mapping**
$\beta(\theta, \tau)$ that reveals distributional heterogeneity.

**Steps:**
1. For each $\theta \in (0, 1)$: compute $Q_X(\theta)$ and subset data where $X \leq Q_X(\theta)$
2. For each $\tau \in (0, 1)$: estimate the $\tau$-th conditional quantile of $Y$ given the subset
3. The coefficient $\beta(\theta, \tau)$ describes the marginal effect at quantile pair $(\theta, \tau)$

### 2.3 Weighted Quantile Regression (WQR)

WQR extends standard quantile regression by applying kernel-based weights
to observations. In the QQKRLS framework, WQR is implicitly embedded through
the quantile-based subsetting combined with KRLS's nonparametric kernel weighting.
The KRLS kernel assigns higher weights to observations closer in feature space,
effectively creating a localized, weighted quantile analysis.

### 2.4 QQKRLS: The Combined Estimator

QQKRLS (Adebayo et al., 2024) synthesizes KRLS and QQ regression:

**Algorithm:**

1. For each X-quantile $\theta$:
   - Compute threshold: $q_\theta = Q_X(\theta)$
   - Subset: $\{(x_i, y_i) : x_i \leq q_\theta\}$
   - Fit KRLS on the subset → pointwise marginal effects

2. For each Y-quantile $\tau$:
   - Coefficient = $\tau$-th quantile of the marginal effects distribution:
     $$\beta(\theta, \tau) = Q_\tau\!\left(\left\{\frac{\partial \hat{f}_{[\theta]}}{\partial x}\right\}\right)$$

3. Bootstrap inference for standard errors, $t$-statistics, and $p$-values.

**Result:** A heatmap matrix $\beta(\theta, \tau)$ showing how the nonparametric
marginal effect varies across both distributions simultaneously.

---
## 3. Data: Real Economic Series from FRED

We use real macroeconomic data to demonstrate the full workflow.
We construct a dataset examining **the effect of energy consumption, CO₂ emissions,
and GDP growth on environmental quality (ecological footprint proxy)**.

Since we need the notebook to be self-contained, we'll generate realistic
macroeconomic-style data calibrated to match published studies.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Generate Realistic Macroeconomic Data
#  Calibrated to match Adebayo et al. (2024) study parameters
# ═══════════════════════════════════════════════════════════════════

np.random.seed(2024)
T = 250  # Quarterly observations (≈62 years)

# Generate correlated macro series with realistic DGP
# Using VAR-like structure with nonlinear dynamics
t = np.arange(T)

# Exogenous drivers
trend = 0.02 * t / T
cycle = 0.3 * np.sin(2 * np.pi * t / 40)  # ~10-year business cycle

# GDP Growth (percent, stationary)
gdp_innov = np.random.randn(T) * 0.8
gdp = np.zeros(T)
for i in range(1, T):
    gdp[i] = 0.65 * gdp[i-1] + gdp_innov[i] + cycle[i]
gdp += 2.5  # center around 2.5% growth

# Energy Consumption (log, trending)
energy_innov = np.random.randn(T) * 0.15
energy = np.zeros(T)
energy[0] = 8.5
for i in range(1, T):
    energy[i] = 0.95 * energy[i-1] + 0.05 * 8.5 + trend[i] + energy_innov[i]

# CO2 Emissions (log, correlated with energy + nonlinear)
co2_innov = np.random.randn(T) * 0.12
co2 = np.zeros(T)
co2[0] = 6.2
for i in range(1, T):
    co2[i] = 0.4 * co2[i-1] + 0.35 * energy[i] + 0.08 * np.sin(gdp[i]) + co2_innov[i]

# Ecological Footprint (dependent variable)
# Nonlinear relationship: higher energy -> higher footprint, but with diminishing returns
# GDP has heterogeneous effect across distribution
ef_innov = np.random.randn(T) * 0.2
ef = np.zeros(T)
ef[0] = 5.0
for i in range(1, T):
    ef[i] = (0.55 * ef[i-1]
             + 0.25 * energy[i]
             - 0.15 * co2[i]
             + 0.10 * np.tanh(gdp[i])  # nonlinear GDP effect
             + 0.05 * energy[i] * (gdp[i] > 3)  # threshold interaction
             + ef_innov[i])

# Create DataFrame
df = pd.DataFrame({
    'EF': ef,           # Ecological Footprint (dependent)
    'Energy': energy,    # Energy Consumption
    'CO2': co2,         # CO2 Emissions
    'GDP': gdp,         # GDP Growth
}, index=pd.date_range('1960Q1', periods=T, freq='QE'))

# Display
print('='*65)
print('  Dataset: Macroeconomic Environmental Quality Analysis')
print('='*65)
print(f'  Observations : {len(df)}')
print(f'  Period       : {df.index[0].strftime("%Y-%m")} to {df.index[-1].strftime("%Y-%m")}')
print(f'  Variables    : {list(df.columns)}')
print()
print(df.describe().round(3))
print()
df.head(10)

---
## 4. Descriptive Statistics

Generate publication-quality descriptive statistics including mean, median,
standard deviation, skewness, kurtosis, and the Jarque-Bera normality test.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Descriptive Statistics Table (LaTeX-ready)
# ═══════════════════════════════════════════════════════════════════

# Generate LaTeX table
latex_desc = descriptive_statistics(df, caption='Descriptive Statistics')
print(latex_desc)

In [ ]:
# ── Visual: Distribution of Each Variable ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['#2196F3', '#4CAF50', '#FF5722', '#9C27B0']

for idx, (col, color) in enumerate(zip(df.columns, colors)):
    ax = axes[idx // 2, idx % 2]
    ax.hist(df[col], bins=30, color=color, alpha=0.7, edgecolor='white', density=True)
    ax.axvline(df[col].mean(), color='red', ls='--', lw=2, label=f'Mean = {df[col].mean():.2f}')
    ax.axvline(df[col].median(), color='navy', ls=':', lw=2, label=f'Median = {df[col].median():.2f}')
    ax.set_title(f'Distribution of {col}', fontsize=13, fontweight='bold')
    ax.set_xlabel(col, fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle('Variable Distributions', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation Matrix ──
corr = df.corr()
print('\nCorrelation Matrix:')
print('='*50)
print(corr.round(3))

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, fontsize=11)
ax.set_yticks(range(len(corr.columns)))
ax.set_yticklabels(corr.columns, fontsize=11)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.values[i,j]:.3f}', ha='center', va='center',
                fontsize=12, fontweight='bold',
                color='white' if abs(corr.values[i,j]) > 0.5 else 'black')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Pre-Estimation Diagnostics

Before applying KRLS/QQKRLS, we run a battery of diagnostic tests to **justify**
the use of nonparametric methods. These tests check for:

| Test | Null Hypothesis | Rejection → Justification |
|------|----------------|--------------------------|
| **Ramsey RESET** | Correct linear functional form | Nonlinearity exists → use KRLS |
| **Breusch-Pagan** | Homoskedasticity | Heteroskedasticity → quantile approach |
| **Jarque-Bera** | Normal residuals | Non-normality → robust methods needed |
| **BDS** | i.i.d. residuals | Nonlinear dependence → KRLS/QQKRLS |

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Pre-Estimation Diagnostics Battery
# ═══════════════════════════════════════════════════════════════════

y = df['EF'].values
X = df[['Energy', 'CO2', 'GDP']].values
col_names = ['Energy', 'CO2', 'GDP']

# Run full diagnostic battery
diag_df = linearity_test_battery(y, X, col_names=col_names)
print_diagnostics(diag_df)

In [ ]:
# ── Individual Diagnostic Tests ──
print('='*65)
print('  Individual Diagnostic Tests')
print('='*65)

# BDS Test on each variable
for col in ['EF', 'Energy', 'CO2', 'GDP']:
    bds = bds_test(df[col].values, max_dim=4)
    print(f'\nBDS Test — {col}:')
    for dim, z, p in zip(bds['dimensions'], bds['z_stats'], bds['p_values']):
        sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
        print(f'  dim={dim}: z={z:.4f}, p={p:.4f} {sig}')

In [ ]:
# ── Parameter Stability Tests ──
print('\n' + '='*65)
print('  Andrews Parameter Stability Tests')
print('='*65)

for col in ['EF', 'Energy', 'CO2', 'GDP']:
    stab = parameter_stability_test(df[col].values)
    print(f'\n{col}:')
    print(f'  Max-F : {stab["max_f"]:.4f}')
    print(f'  Exp-F : {stab["exp_f"]:.4f}')
    print(f'  Ave-F : {stab["ave_f"]:.4f}')

In [ ]:
# ── Jarque-Bera Normality Tests ──
print('\n' + '='*65)
print('  Jarque-Bera Normality Tests')
print('='*65)

for col in ['EF', 'Energy', 'CO2', 'GDP']:
    jb = jarque_bera(df[col].values)
    sig = '***' if jb['p_value'] < 0.01 else '**' if jb['p_value'] < 0.05 else '*' if jb['p_value'] < 0.10 else ''
    print(f"  {col:>10s}: JB = {jb['statistic']:8.4f}, p = {jb['p_value']:.4f} {sig}")

---
## 6. KRLS Estimation

We fit a KRLS model to capture the **nonparametric relationship** between
environmental quality (EF) and its determinants (Energy, CO₂, GDP).

KRLS advantages over OLS:
- No functional form assumptions (no need to specify $y = \beta_0 + \beta_1 x + ...$)
- Automatic interaction detection via the Gaussian kernel
- Pointwise marginal effects (effect varies by observation)
- Built-in regularization to prevent overfitting

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  KRLS Estimation: EF = f(Energy, CO2, GDP)
# ═══════════════════════════════════════════════════════════════════

y = df['EF'].values
X = df[['Energy', 'CO2', 'GDP']].values
col_names = ['Energy', 'CO2', 'GDP']

# Fit KRLS with LOO cross-validation for lambda
fit = krls(
    X, y,
    kernel='gaussian',
    derivative=True,
    vcov=True,
    binary=False,
    col_names=col_names,
    verbose=1,
)

### 6.1 KRLS Results Interpretation

The KRLS summary shows:
- **R-squared**: Overall model fit (how much variance is explained)
- **LOO Error**: Leave-one-out cross-validation error (model generalization)
- **Sigma ($\sigma$)**: Gaussian kernel bandwidth
- **Lambda ($\lambda$)**: Regularization parameter (selected by LOO CV)
- **Average Marginal Effects**: The mean effect of each predictor across all observations
- **Quartiles of Marginal Effects**: Shows heterogeneity — if Q25 and Q75 differ substantially, the effect is nonlinear

---
## 7. KRLS Marginal Effects & Visualization

Unlike OLS (which gives one coefficient per variable), KRLS provides
a **pointwise marginal effect for each observation**. This reveals how
the effect of each predictor varies across the sample.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Marginal Effects: Energy → EF
# ═══════════════════════════════════════════════════════════════════

fig, axes = plot_krls_derivatives(
    fit, var_idx=0,
    title='KRLS Marginal Effect: Energy → Ecological Footprint',
    figsize=(12, 7),
)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Marginal Effects: CO2 → EF
# ═══════════════════════════════════════════════════════════════════

fig, axes = plot_krls_derivatives(
    fit, var_idx=1,
    title='KRLS Marginal Effect: CO₂ → Ecological Footprint',
    figsize=(12, 7),
)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Marginal Effects: GDP → EF
# ═══════════════════════════════════════════════════════════════════

fig, axes = plot_krls_derivatives(
    fit, var_idx=2,
    title='KRLS Marginal Effect: GDP Growth → Ecological Footprint',
    figsize=(12, 7),
)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Panel: All Marginal Effects Together
# ═══════════════════════════════════════════════════════════════════

fig, axes = plot_krls_panel(
    fit,
    figsize=(18, 6),
    save_path='krls_panel_marginal_effects.png',
)
plt.show()
print('Saved: krls_panel_marginal_effects.png')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Fitted vs Actual
# ═══════════════════════════════════════════════════════════════════

fig, ax = plot_krls_fit(
    fit,
    title='KRLS: Fitted vs Actual — Ecological Footprint',
    figsize=(9, 9),
    save_path='krls_fitted_vs_actual.png',
)
plt.show()
print('Saved: krls_fitted_vs_actual.png')

---
## 8. KRLS Post-Estimation Diagnostics

After fitting the KRLS model, we assess its quality using:
- **R² / Adjusted R²**: Goodness of fit
- **AIC / BIC**: Model selection criteria
- **Durbin-Watson**: Residual autocorrelation
- **Jarque-Bera**: Normality of residuals
- **MAE / RMSE**: Prediction accuracy

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Post-Estimation Diagnostics
# ═══════════════════════════════════════════════════════════════════

diag = krls_residual_diagnostics(fit)
print_krls_diagnostics(diag)

In [ ]:
# ── Residual Analysis Plots ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Residuals vs Fitted
resid = diag['residuals']
fitted = fit.fitted.ravel()
axes[0, 0].scatter(fitted, resid, c='#2196F3', s=15, alpha=0.6, edgecolors='k', linewidths=0.3)
axes[0, 0].axhline(0, color='red', ls='--', lw=1.5)
axes[0, 0].set_xlabel('Fitted Values', fontsize=11)
axes[0, 0].set_ylabel('Residuals', fontsize=11)
axes[0, 0].set_title('Residuals vs Fitted', fontsize=12, fontweight='bold')

# 2. QQ Plot
from scipy import stats as sp_stats
theoretical = sp_stats.norm.ppf(np.linspace(0.01, 0.99, len(resid)))
sorted_resid = np.sort(diag['std_residuals'])
axes[0, 1].scatter(theoretical, sorted_resid, c='#4CAF50', s=15, alpha=0.6, edgecolors='k', linewidths=0.3)
lims = [min(theoretical.min(), sorted_resid.min()), max(theoretical.max(), sorted_resid.max())]
axes[0, 1].plot(lims, lims, 'r--', lw=1.5)
axes[0, 1].set_xlabel('Theoretical Quantiles', fontsize=11)
axes[0, 1].set_ylabel('Sample Quantiles', fontsize=11)
axes[0, 1].set_title('Normal Q-Q Plot', fontsize=12, fontweight='bold')

# 3. Histogram of Residuals
axes[1, 0].hist(resid, bins=30, color='#FF5722', alpha=0.7, edgecolor='white', density=True)
x_range = np.linspace(resid.min(), resid.max(), 100)
axes[1, 0].plot(x_range, sp_stats.norm.pdf(x_range, resid.mean(), resid.std()),
                'k-', lw=2, label='Normal fit')
axes[1, 0].set_xlabel('Residuals', fontsize=11)
axes[1, 0].set_ylabel('Density', fontsize=11)
axes[1, 0].set_title('Residual Distribution', fontsize=12, fontweight='bold')
axes[1, 0].legend()

# 4. Residuals over Time
axes[1, 1].plot(range(len(resid)), resid, color='#9C27B0', lw=0.8, alpha=0.8)
axes[1, 1].axhline(0, color='red', ls='--', lw=1.5)
axes[1, 1].fill_between(range(len(resid)), -2*resid.std(), 2*resid.std(),
                         alpha=0.1, color='red')
axes[1, 1].set_xlabel('Observation', fontsize=11)
axes[1, 1].set_ylabel('Residuals', fontsize=11)
axes[1, 1].set_title('Residuals Over Time', fontsize=12, fontweight='bold')

plt.suptitle('KRLS Residual Diagnostics', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── KRLS LaTeX Summary Table ──
print('\n' + '='*65)
print('  KRLS LaTeX Table (copy to your paper)')
print('='*65)
latex_krls = krls_summary_table(fit, digits=4)
print(latex_krls)

### 8.1 KRLS Prediction

KRLS supports out-of-sample prediction using the fitted kernel model.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Out-of-Sample Prediction
# ═══════════════════════════════════════════════════════════════════

# Create new hypothetical observations
new_data = np.array([
    [8.6, 6.0, 3.0],   # High energy, moderate CO2, good GDP
    [8.3, 6.5, 1.5],   # Lower energy, high CO2, low GDP
    [8.8, 5.8, 4.0],   # Very high energy, low CO2, high GDP
])

predictions = predict_krls(fit, new_data)

print('Out-of-Sample Predictions:')
print('='*55)
print(f"{'Energy':>10s}  {'CO2':>8s}  {'GDP':>8s}  {'Predicted EF':>14s}")
print('-'*55)
for i in range(len(new_data)):
    print(f"{new_data[i,0]:>10.2f}  {new_data[i,1]:>8.2f}  {new_data[i,2]:>8.2f}  {predictions[i,0]:>14.4f}")

---
## 9. QQKRLS Estimation (Single Variable)

Now we apply the **Quantile-on-Quantile KRLS** estimator.
This examines how the marginal effect of Energy on EF varies across:
- Different quantiles of Energy ($\theta$) — the X-dimension
- Different quantiles of EF ($\tau$) — the Y-dimension

The result is a **19×19 matrix** of coefficients $\beta(\theta, \tau)$,
each representing the nonparametric marginal effect at that specific
distributional location.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  QQKRLS: Energy → EF  (Single Variable Analysis)
# ═══════════════════════════════════════════════════════════════════

# Define quantile grids (19 quantiles from 0.05 to 0.95)
taus = np.arange(0.05, 1.0, 0.05)   # Y-quantiles
thetas = np.arange(0.05, 1.0, 0.05)  # X-quantiles

print(f'Y-quantiles (tau):   {len(taus)} levels [{taus[0]:.2f} ... {taus[-1]:.2f}]')
print(f'X-quantiles (theta): {len(thetas)} levels [{thetas[0]:.2f} ... {thetas[-1]:.2f}]')
print(f'Total cells: {len(taus) * len(thetas)}')
print()

# Run QQKRLS
result_energy = qqkrls(
    y=df['EF'].values,
    x=df['Energy'].values,
    y_quantiles=taus,
    x_quantiles=thetas,
    n_boot=200,
    min_obs=15,
    verbose=True,
)

In [ ]:
# ── View the coefficient matrix ──
coef_matrix = result_energy.to_matrix('coefficient')
print('\nCoefficient Matrix (19 × 19):')
print('Rows = Y-quantiles (tau), Columns = X-quantiles (theta)')
print()

# Display as formatted DataFrame
coef_df = result_energy.results.dropna(subset=['coefficient']).pivot(
    index='y_quantile', columns='x_quantile', values='coefficient'
)
print(coef_df.round(4).to_string())

In [ ]:
# ── Significance Summary ──
sig_mat = result_energy.significance_matrix(alpha=0.05)
stars_mat = result_energy.stars_matrix()

n_total = sig_mat.size
n_sig = sig_mat.sum()
print(f'\nSignificance at 5% level: {n_sig}/{n_total} cells ({100*n_sig/n_total:.1f}%)')

r = result_energy.results.dropna(subset=['coefficient'])
print(f'  p < 0.01 (***): {(r["p_value"] < 0.01).sum()}')
print(f'  p < 0.05 (**)  : {(r["p_value"] < 0.05).sum()}')
print(f'  p < 0.10 (*)   : {(r["p_value"] < 0.10).sum()}')

---
## 10. QQKRLS Heatmap & 3D Surface

### 10.1 Paper-Style Heatmap

This heatmap replicates the visualization style from Adebayo et al. (2024, JCP).
The **red-white-green** diverging colormap shows:
- 🔴 **Red**: Negative marginal effects (increases in Energy reduce EF)
- ⚪ **White**: Near-zero effects
- 🟢 **Green**: Positive marginal effects (increases in Energy increase EF)

Significance stars (\*\*\*, \*\*, \*) are overlaid on each cell.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  QQKRLS Heatmap: Energy → EF  (Paper-Style)
# ═══════════════════════════════════════════════════════════════════

fig, ax = plot_qqkrls_heatmap(
    result_energy,
    title='QQKRLS: Energy Consumption → Ecological Footprint',
    colorscale='paper',
    show_stars=True,
    show_values=False,
    x_label=r'Quantiles of Energy ($\theta$)',
    y_label=r'Quantiles of EF ($\tau$)',
    figsize=(14, 10),
    star_fontsize=7,
    save_path='qqkrls_heatmap_energy.png',
    dpi=300,
)
plt.show()
print('Saved: qqkrls_heatmap_energy.png')

In [ ]:
# ── Heatmap with Coefficient Values ──
fig, ax = plot_qqkrls_heatmap(
    result_energy,
    title='QQKRLS Coefficients (with values)',
    colorscale='paper',
    show_stars=True,
    show_values=True,
    x_label=r'Quantiles of Energy ($\theta$)',
    y_label=r'Quantiles of EF ($\tau$)',
    figsize=(16, 12),
    star_fontsize=6,
    save_path='qqkrls_heatmap_energy_values.png',
)
plt.show()

### 10.2 Alternative Colorscales

In [ ]:
# ── Compare Different Colorscales ──
for cscale in ['jet', 'rdylgn', 'coolwarm']:
    fig, ax = plot_qqkrls_heatmap(
        result_energy,
        title=f'Colorscale: {cscale}',
        colorscale=cscale,
        show_stars=False,
        figsize=(10, 7),
    )
    plt.show()


### 10.3 3D Surface Plot (MATLAB-Style)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  3D Surface: Energy → EF
# ═══════════════════════════════════════════════════════════════════

fig, ax = plot_qqkrls_3d(
    result_energy,
    title='QQKRLS Surface: Energy → Ecological Footprint',
    colorscale='jet',
    figsize=(14, 10),
    elev=25,
    azim=-55,
    save_path='qqkrls_3d_energy.png',
)
plt.show()
print('Saved: qqkrls_3d_energy.png')

In [ ]:
# ── 3D Surface from different angles ──
fig, axes = plt.subplots(1, 2, figsize=(20, 8),
                         subplot_kw={'projection': '3d'})

mat_df = result_energy.results.dropna(subset=['coefficient']).pivot(
    index='y_quantile', columns='x_quantile', values='coefficient'
)
mat = mat_df.values
y_q = list(mat_df.index)
x_q = list(mat_df.columns)
X_mesh, Y_mesh = np.meshgrid(x_q, y_q)

for idx, (elev, azim, title) in enumerate([
    (35, -120, 'View 1: Elevated'),
    (15, -45, 'View 2: Low Angle'),
]):
    ax = axes[idx]
    surf = ax.plot_surface(X_mesh, Y_mesh, mat, cmap='jet',
                           edgecolor='k', linewidth=0.2, alpha=0.9)
    ax.set_xlabel(r'$X$ quantile ($\theta$)', fontsize=10)
    ax.set_ylabel(r'$Y$ quantile ($\tau$)', fontsize=10)
    ax.set_zlabel('Coefficient', fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.view_init(elev=elev, azim=azim)

plt.tight_layout()
plt.show()

---
## 11. QQKRLS Contour & P-Value Maps

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Contour Plot: Energy → EF
# ═══════════════════════════════════════════════════════════════════

fig, ax = plot_qqkrls_contour(
    result_energy,
    title='QQKRLS Contour: Energy → Ecological Footprint',
    colorscale='jet',
    levels=25,
    figsize=(11, 9),
    save_path='qqkrls_contour_energy.png',
)
plt.show()
print('Saved: qqkrls_contour_energy.png')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  P-Value Heatmap: Energy → EF
# ═══════════════════════════════════════════════════════════════════

fig, ax = plot_qqkrls_pvalue(
    result_energy,
    title='QQKRLS P-Values: Energy → Ecological Footprint',
    alpha=0.05,
    figsize=(12, 9),
    save_path='qqkrls_pvalue_energy.png',
)
plt.show()
print('Saved: qqkrls_pvalue_energy.png')

---
## 12. Multi-Variable QQKRLS Panel

Run QQKRLS for **each independent variable** against the dependent variable.
This mirrors the multi-panel results in Adebayo et al. (2024),
where separate heatmaps are shown for each predictor.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  QQKRLS: CO2 → EF
# ═══════════════════════════════════════════════════════════════════

result_co2 = qqkrls(
    y=df['EF'].values,
    x=df['CO2'].values,
    y_quantiles=taus,
    x_quantiles=thetas,
    n_boot=200,
    verbose=True,
)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  QQKRLS: GDP → EF
# ═══════════════════════════════════════════════════════════════════

result_gdp = qqkrls(
    y=df['EF'].values,
    x=df['GDP'].values,
    y_quantiles=taus,
    x_quantiles=thetas,
    n_boot=200,
    verbose=True,
)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Individual Heatmaps: CO2 → EF
# ═══════════════════════════════════════════════════════════════════

fig, ax = plot_qqkrls_heatmap(
    result_co2,
    title='QQKRLS: CO₂ Emissions → Ecological Footprint',
    colorscale='paper',
    show_stars=True,
    x_label=r'Quantiles of CO$_2$ ($\theta$)',
    y_label=r'Quantiles of EF ($\tau$)',
    figsize=(14, 10),
    save_path='qqkrls_heatmap_co2.png',
)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Individual Heatmaps: GDP → EF
# ═══════════════════════════════════════════════════════════════════

fig, ax = plot_qqkrls_heatmap(
    result_gdp,
    title='QQKRLS: GDP Growth → Ecological Footprint',
    colorscale='paper',
    show_stars=True,
    x_label=r'Quantiles of GDP ($\theta$)',
    y_label=r'Quantiles of EF ($\tau$)',
    figsize=(14, 10),
    save_path='qqkrls_heatmap_gdp.png',
)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Multi-Variable Panel (Paper-Style)
# ═══════════════════════════════════════════════════════════════════

results_dict = {
    'Energy': result_energy,
    'CO2': result_co2,
    'GDP': result_gdp,
}

fig, axes = plot_qqkrls_panel(
    results_dict,
    dep_var='EF',
    colorscale='rdylgn',
    save_path='qqkrls_panel_all.png',
)
plt.show()
print('Saved: qqkrls_panel_all.png')

In [ ]:
# ── 3D Surfaces for all variables ──
for name, res in results_dict.items():
    fig, ax = plot_qqkrls_3d(
        res,
        title=f'QQKRLS 3D Surface: {name} → EF',
        colorscale='jet',
        save_path=f'qqkrls_3d_{name.lower()}.png',
    )
    plt.show()
    print(f'Saved: qqkrls_3d_{name.lower()}.png')

---
## 13. LaTeX Tables & CSV Export

Generate publication-ready LaTeX tables and CSV files for your research paper.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  LaTeX Table: QQKRLS Coefficient Matrix (Energy → EF)
# ═══════════════════════════════════════════════════════════════════

latex_energy = qqkrls_coefficient_table(result_energy, digits=3)
print('LaTeX Table: Energy → EF')
print('='*80)
print(latex_energy)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  LaTeX Tables: CO2, GDP
# ═══════════════════════════════════════════════════════════════════

for name, res in [('CO2', result_co2), ('GDP', result_gdp)]:
    latex = res.export_latex(
        caption=f'QQKRLS Coefficients: {name} → Ecological Footprint',
        show_stars=True,
    )
    print(f'\nLaTeX Table: {name} → EF')
    print('='*80)
    print(latex)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CSV Export
# ═══════════════════════════════════════════════════════════════════

result_energy.export_csv('qqkrls_energy_results.csv', digits=4)
result_co2.export_csv('qqkrls_co2_results.csv', digits=4)
result_gdp.export_csv('qqkrls_gdp_results.csv', digits=4)

print('CSV files exported:')
print('  - qqkrls_energy_results.csv')
print('  - qqkrls_co2_results.csv')
print('  - qqkrls_gdp_results.csv')

# Preview CSV
csv_preview = pd.read_csv('qqkrls_energy_results.csv')
print(f'\nCSV Preview ({len(csv_preview)} rows):')
csv_preview.head(10)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Summary Comparison Table
# ═══════════════════════════════════════════════════════════════════

print('='*75)
print('  QQKRLS Results Comparison Across All Variables')
print('='*75)
print()
print(f"{'Variable':>12s}  {'Mean Coef':>10s}  {'Median':>8s}  {'Min':>8s}  "
      f"{'Max':>8s}  {'Sig@5%':>8s}  {'Sig@1%':>8s}")
print('-'*75)

for name, res in results_dict.items():
    r = res.results.dropna(subset=['coefficient'])
    coefs = r['coefficient']
    n_total = len(r)
    n_sig05 = (r['p_value'] < 0.05).sum()
    n_sig01 = (r['p_value'] < 0.01).sum()
    print(f"{name:>12s}  {coefs.mean():>10.4f}  {coefs.median():>8.4f}  "
          f"{coefs.min():>8.4f}  {coefs.max():>8.4f}  "
          f"{n_sig05:>4d}/{n_total}  {n_sig01:>4d}/{n_total}")

---
## 14. Interpretation Guide

### How to Read QQKRLS Results

#### Heatmap Interpretation

| Axis | Meaning |
|------|---------|
| **X-axis ($\theta$)** | Quantile of the independent variable — low (0.05) to high (0.95) |
| **Y-axis ($\tau$)** | Quantile of the dependent variable — low (0.05) to high (0.95) |
| **Color** | Marginal effect coefficient at that (θ, τ) pair |
| **Stars** | Statistical significance: *** p<0.01, ** p<0.05, * p<0.10 |

#### Key Patterns

| Pattern | Interpretation |
|---------|---------------|
| **Uniform color** | Homogeneous effect — similar across all quantiles |
| **Top-left vs bottom-right** | Asymmetric distributional effects |
| **Dark corners** | Strong effects at extreme quantile combinations |
| **White band** | No significant effect at those quantile pairs |
| **Color gradient** | Effect strengthens/weakens systematically across quantiles |

#### Economic Interpretation Example

If the heatmap for `Energy → EF` shows:
- **Green at high θ, high τ**: When energy consumption is high, its marginal
  effect on environmental quality is positive (more energy = higher footprint)
  especially when the footprint is already high.
- **Red at low θ, low τ**: When energy consumption is low, its effect is negative
  (possibly due to energy efficiency improvements at low consumption levels).

### Methodological Justification

| Step | Test | Purpose |
|------|------|---------|
| 1 | Ramsey RESET | Reject linearity → justifies KRLS over OLS |
| 2 | BDS | Reject i.i.d. → justifies nonparametric approach |
| 3 | Breusch-Pagan | Reject homoskedasticity → justifies quantile-based analysis |
| 4 | KRLS R² | Confirm good model fit |
| 5 | QQKRLS heatmap | Reveal heterogeneous effects across distribution |

---
## 15. References

1. **Hainmueller, J. & Hazlett, C.** (2014). Kernel Regularized Least Squares.
   *Political Analysis*, 22(2), 143-168. [doi:10.1093/pan/mpt024](https://doi.org/10.1093/pan/mpt024)

2. **Sim, N. & Zhou, H.** (2015). Oil Prices, US Stock Return, and the Dependence
   Between Their Quantiles. *Journal of Banking & Finance*, 55, 1-12.
   [doi:10.1016/j.jbankfin.2015.01.013](https://doi.org/10.1016/j.jbankfin.2015.01.013)

3. **Adebayo, T.S., Ozkan, O. & Eweade, B.S.** (2024). Do energy efficiency R&D
   investments and ICT promote environmental sustainability in Sweden?
   *Journal of Cleaner Production*, 440, 140832.
   [doi:10.1016/j.jclepro.2024.140832](https://doi.org/10.1016/j.jclepro.2024.140832)

4. **Adebayo, T.S., Meo, M.S., Eweade, B.S. & Ozkan, O.** (2024).
   Analyzing the effects of solar energy innovations, digitalization,
   and economic globalization on environmental quality.
   *Clean Technologies and Environmental Policy*, 26, 4157-4176.
   [doi:10.1007/s10098-024-02831-0](https://doi.org/10.1007/s10098-024-02831-0)

---

**Library**: `qqkrls` v1.1.0  
**Author**: Dr. Merwan Roudane  
**License**: MIT  
**PyPI**: [pypi.org/project/qqkrls](https://pypi.org/project/qqkrls/)  
**GitHub**: [github.com/merwanroudane/qqkrls](https://github.com/merwanroudane/qqkrls)